In [1]:
import sys
import time
import gc
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from src import config
from src.data import load

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
print("Repo root:", ROOT)


Repo root: c:\POST-BUILD1.O-PROJECT\DemandIQ


In [2]:
EXPECTED = [ 
    "calendar.csv", "sales_train_validation.csv", 
    "sales_train_evaluation.csv", 
    "sell_prices.csv", "sample_submission.csv"
    ]

missing_files = [f for f in EXPECTED if not (config.RAW_DIR / f).exists()]
assert not missing_files, f"Missing from data/raw: {missing_files}"

for f in EXPECTED:
    print(f"{f:<30}{(config.RAW_DIR / f).stat().st_size / 1024**2:>10.1f} MB on disk")

calendar.csv                         0.1 MB on disk
sales_train_validation.csv         114.4 MB on disk
sales_train_evaluation.csv         116.1 MB on disk
sell_prices.csv                    194.0 MB on disk
sample_submission.csv                5.0 MB on disk


In [3]:
print("RAW_DIR :", config.RAW_DIR.resolve())
print("exists  :", config.RAW_DIR.exists())

RAW_DIR : C:\POST-BUILD1.O-PROJECT\DemandIQ\data\raw
exists  : True


In [4]:
print("Looking inside:", config.RAW_DIR.resolve())
print("Folder exists?:", config.RAW_DIR.exists())

if config.RAW_DIR.exists():
    print("Files currently inside:", list(config.RAW_DIR.iterdir()))

Looking inside: C:\POST-BUILD1.O-PROJECT\DemandIQ\data\raw
Folder exists?: True
Files currently inside: [WindowsPath('C:/POST-BUILD1.O-PROJECT/DemandIQ/data/raw/.gitkeep'), WindowsPath('C:/POST-BUILD1.O-PROJECT/DemandIQ/data/raw/calendar.csv'), WindowsPath('C:/POST-BUILD1.O-PROJECT/DemandIQ/data/raw/sales_train_evaluation.csv'), WindowsPath('C:/POST-BUILD1.O-PROJECT/DemandIQ/data/raw/sales_train_validation.csv'), WindowsPath('C:/POST-BUILD1.O-PROJECT/DemandIQ/data/raw/sample_submission.csv'), WindowsPath('C:/POST-BUILD1.O-PROJECT/DemandIQ/data/raw/sell_prices.csv')]


In [5]:
t = time.time()
calendar = load.read_calendar()
sales = load.read_sales(version="evaluation")
prices = load.read_prices()
print(f"Loaded in {time.time() - t:.1f}s")
day_cols = [c for c in sales.columns if c.startswith("d_")]

Loaded in 20.3s


In [ ]:
dims = pd.DataFrame({
    "file": ["calendar", "sales (evaluation)", "sell_prices"],
    "rows": [len(calendar), len(sales), len(prices)],
    "columns": [calendar.shape[1], sales.shape[1], prices.shape[1]],
    "memory_MB": [round(load.memory_mb(d), 1) for d in (calendar, sales, prices)],
})

print(dims.to_string(index=False))
print("\nDay columns in sales:", len(day_cols), "|", day_cols[0], "->", day_cols[-1])

              file    rows  columns  memory_MB
          calendar    1969       14        0.1
sales (evaluation)   30490     1947      114.2
       sell_prices 6841121        4       58.8

Day columns in sales: 1941 | d_1 -> d_1941


: 

In [ ]:
# columns and types
print("CALENDAR\n", calendar.dtypes, "\n")
print("SALES, identifier columns\n", sales.iloc[:, :6].dtypes, "\n")
print("SALES, dtype counts across all columns\n", sales.dtypes.value_counts(), "\n")
print("PRICES\n", prices.dtypes)
calendar.head()

CALENDAR
 date            datetime64[us]
wm_yr_wk                 int16
weekday               category
wday                      int8
month                     int8
year                     int16
d                          str
event_name_1          category
event_type_1          category
event_name_2          category
event_type_2          category
snap_CA                   int8
snap_TX                   int8
snap_WI                   int8
dtype: object 

SALES, identifier columns
 id               str
item_id     category
dept_id     category
cat_id      category
store_id    category
state_id    category
dtype: object 

SALES, dtype counts across all columns
 int16       1941
str            1
category       1
category       1
category       1
category       1
category       1
Name: count, dtype: int64 

PRICES
 store_id      category
item_id       category
wm_yr_wk         int16
sell_price     float32
dtype: object


,date,wm_yr_wk,weekday,wday,month,year,d,event_name_1,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI
0,2011-01-29,11101,Saturday,1,1,2011,d_1,NaN,NaN,NaN,NaN,0,0,0
1,2011-01-30,11101,Sunday,2,1,2011,d_2,NaN,NaN,NaN,NaN,0,0,0
2,2011-01-31,11101,Monday,3,1,2011,d_3,NaN,NaN,NaN,NaN,0,0,0
3,2011-02-01,11101,Tuesday,4,2,2011,d_4,NaN,NaN,NaN,NaN,1,1,0
4,2011-02-02,11101,Wednesday,5,2,2011,d_5,NaN,NaN,NaN,NaN,1,0,1


: 

In [ ]:
sales.iloc[:3, :10]

,id,item_id,dept_id,cat_id,store_id,state_id,d_1,d_2,d_3,d_4
0,HOBBIES_1_001_CA_1_evaluation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0
1,HOBBIES_1_002_CA_1_evaluation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0
2,HOBBIES_1_003_CA_1_evaluation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,0,0,0,0


: 

In [ ]:
prices.head()

,store_id,item_id,wm_yr_wk,sell_price
0,CA_1,HOBBIES_1_001,11325,9.58
1,CA_1,HOBBIES_1_001,11326,9.58
2,CA_1,HOBBIES_1_001,11327,8.26
3,CA_1,HOBBIES_1_001,11328,8.26
4,CA_1,HOBBIES_1_001,11329,8.26


: 

In [ ]:
# missing values and duplicates

reports = {
    "calendar": load.quality_report(calendar, key_cols=["date"]),
    "sales": load.quality_report(sales, key_cols=["id"]),
    "prices": load.quality_report(prices, key_cols=["store_id", "item_id", "wm_yr_wk"]),

}

for name, rep in reports.items():
    print(f"---{name} ---")
    for k, v in rep.items():
        print(f" {k}: {v}")

---calendar ---
 rows: 1969
 columns: 14
 columns_with_missing: {'event_name_1': 1807, 'event_type_1': 1807, 'event_name_2': 1964, 'event_type_2': 1964}
 full_row_duplicates: 0
 key_duplicates: 0
---sales ---
 rows: 30490
 columns: 1947
 columns_with_missing: {}
 full_row_duplicates: 0
 key_duplicates: 0
---prices ---
 rows: 6841121
 columns: 4
 columns_with_missing: {}
 full_row_duplicates: 0
 key_duplicates: 0


: 

In [ ]:
# price-coverage check

n_items = prices["item_id"].nunique()
n_weeks = calendar["wm_yr_wk"].nunique()
n_stores = prices["store_id"].nunique()

print(f"Items {n_items}, stores {n_stores}, weeks {n_weeks}")
print(f"Rows if every item were priced in every store-week: {n_items * n_stores * n_weeks:,}")
print(f"Actual price rows: {len(prices):,}")
print(f"Share of store-item-weeks with a price row: "
      f"{len(prices) / (n_items * n_stores * n_weeks):.1%}")

Items 3049, stores 10, weeks 282
Rows if every item were priced in every store-week: 8,598,180
Actual price rows: 6,841,121
Share of store-item-weeks with a price row: 79.6%


: 

In [ ]:
#cross-file consistency

# every sales day column exists in the calendar
cal_days = set(calendar["d"])
print("Sales day columns all in calendar:", set(day_cols) <= cal_days)

# calendar dates are contiguous (no skipped days)
gaps = calendar["date"].diff().dropna()
print("Calendar contiguous daily:", (gaps == pd.Timedelta(days=1)).all())

# date coverage of the sales history
cal_idx = calendar.set_index("d")["date"]
print("Sales history:", cal_idx[day_cols[0]].date(), "->", cal_idx[day_cols[-1]].date())

# every price week exists in the calendar
print("Price weeks all in calendar:", set(prices["wm_yr_wk"]) <= set(calendar["wm_yr_wk"]))

# ids in prices vs sales
print("Items match:", set(prices["item_id"].unique()) == set(sales["item_id"].unique()))
print("Stores match:", set(prices["store_id"].unique()) == set(sales["store_id"].unique()))


# sales value range (guards the int16 decision)
print("Sales min / max:", sales[day_cols].min().min(), "/", sales[day_cols].max().max())

Sales day columns all in calendar: True
Calendar contiguous daily: True
Sales history: 2011-01-29 -> 2016-05-22
Price weeks all in calendar: True
Items match: True
Stores match: True
Sales min / max: 0 / 763


: 

In [ ]:
# validation file vs evaluation file

val = load.read_sales(version="validation")
val_days = [c for c in val.columns if c.startswith("d_")]

print("Example ids:", val["id"].iloc[0], "|", sales["id"].iloc[0])
val_ids = val["id"].str.replace("_validation$", "", regex=True)
eval_ids = sales["id"].str.replace("_evaluation$", "", regex=True)


same_order = (val_ids.to_numpy() == eval_ids.to_numpy()).all()
same_values = (val[val_days].to_numpy() == sales[val_days].to_numpy()).all()
print("Series in same order:", same_order, "| shared days identical:", same_values,
      "| validation days:", len(val_days))
del val; gc.collect()

Example ids: HOBBIES_1_001_CA_1_validation | HOBBIES_1_001_CA_1_evaluation
Series in same order: True | shared days identical: True | validation days: 1913


0

: 

In [ ]:
naive = {}
for label, fname in [("calendar", "calendar.csv"),
                     ("sales", "sales_train_evaluation.csv"),
                     ("prices", "sell_prices.csv")]:
    df = pd.read_csv(config.RAW_DIR / fname)
    naive[label] = load.memory_mb(df)
    del df; gc.collect()

opt = {"calendar": load.memory_mb(calendar), "sales": load.memory_mb(sales),
       "prices": load.memory_mb(prices)}
cmp = pd.DataFrame({"naive_MB": naive, "optimised_MB": opt})
cmp["reduction"] = (1 - cmp["optimised_MB"] / cmp["naive_MB"]).map("{:.0%}".format)
print(cmp.round(1))
print("\nTotal naive MB:", round(sum(naive.values())), "| optimised MB:", round(sum(opt.values())))

          naive_MB  optimised_MB reduction
calendar       0.3           0.1       73%
sales        454.7         114.2       75%
prices       318.1          58.8       82%

Total naive MB: 773 | optimised MB: 173


: 

In [ ]:
n_series, n_days = len(sales), len(day_cols)
rows = n_series * n_days

# bytes per row under a compact dtype plan
plan = {"item code (int16)": 2, "store/dept/cat/state codes (4 x int8)": 4,
        "day index (int16)": 2, "sales (int16)": 2, "price (float32)": 4}
bpr = sum(plan.values())
print(f"Long-format rows: {rows:,}")
print(f"Bytes per row: {bpr}")
print(f"Projected size: {rows * bpr / 1024**3:.2f} GB (estimate; excludes calendar features)")
print(f"With a 64-bit date column added: {rows * (bpr + 8) / 1024**3:.2f} GB")

Long-format rows: 59,181,090
Bytes per row: 14
Projected size: 0.77 GB (estimate; excludes calendar features)
With a 64-bit date column added: 1.21 GB


: 

: 